# Pinterest 원룸 인테리어 보드 크롤링

[Pinterest 보드](https://kr.pinterest.com/718tndydlf44/%EC%9B%90%EB%A3%B8-%EC%9D%B8%ED%85%8C%EB%A6%AC%EC%96%B4/) (490 Pins)에서 이미지를 수집합니다.

- **선택자:** `div[data-test-id="pin-with-alt-text"] img`
- **고해상도:** `srcset` → `/originals/` URL 변환
- **백그라운드:** Chrome 최소화 (headless 차단)
- **한 줄씩 다운로드** — 화면 그리드 열 수를 Y좌표로 자동 감지 → 없으면 스크롤

In [1]:
# 크롤링 의존성: selenium(브라우저), webdriver-manager(ChromeDriver), requests(이미지 다운로드)
%pip install selenium webdriver-manager requests

Note: you may need to restart the kernel to use updated packages.


In [8]:
import time  # 스크롤·대기
import hashlib  # URL → 파일명 해시
from pathlib import Path  # 저장 경로
from urllib.parse import urlparse  # URL에서 확장자 추출

import requests  # HTTP 이미지 다운로드
from selenium import webdriver  # Chrome 브라우저 제어
from selenium.webdriver.chrome.service import Service  # ChromeDriver 서비스
from selenium.webdriver.chrome.options import Options  # headless/최소화 등 옵션
from selenium.webdriver.common.by import By  # CSS 셀렉터
from selenium.webdriver.support.ui import WebDriverWait  # 요소 로딩 대기
from selenium.webdriver.support import expected_conditions as EC  # 대기 조건
from webdriver_manager.chrome import ChromeDriverManager  # ChromeDriver 자동 설치

In [9]:
# === 설정 ===
BOARD_URL = "https://kr.pinterest.com/718tndydlf44/%EC%9B%90%EB%A3%B8-%EC%9D%B8%ED%85%8C%EB%A6%AC%EC%96%B4/"  # Pinterest 보드 URL
SAVE_DIR = Path("images/pinterest_원룸_인테리어")  # 이미지 저장 폴더
SAVE_DIR.mkdir(parents=True, exist_ok=True)

PIN_IMG_SELECTOR = 'div[data-test-id="pin-with-alt-text"] img'  # 핀 이미지 셀렉터
ROW_Y_TOLERANCE = 50       # 같은 줄(높이) 허용(px)
COLUMN_X_TOLERANCE = 40    # 같은 열(너비) 허용(px) — 열 개수 = 동적 row_size
SCROLL_PAUSE = 2.0         # 스크롤 후 대기(초)
PAGE_LOAD_WAIT = 30        # 핀 로딩 대기(초)
MAX_IMAGES = 300           # 최대 다운로드 개수

In [10]:
def create_driver():
    """Chrome 드라이버 생성 (최소화 모드, 봇 탐지 우회)."""
    options = Options()
    options.add_argument("--start-minimized")  # headless 대신 최소화
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options,
    )
    driver.execute_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    )
    return driver


def wait_for_pins(driver, timeout=PAGE_LOAD_WAIT):
    """Pinterest 보드 핀이 로드될 때까지 대기."""
    print(f"보드 로딩 대기... (최대 {timeout}초)")
    try:
        WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, PIN_IMG_SELECTOR))
        )
    except Exception:
        raise TimeoutError(
            f"핀을 찾지 못했습니다. title={driver.title}, url={driver.current_url}"
        ) from None
    n = len(driver.find_elements(By.CSS_SELECTOR, PIN_IMG_SELECTOR))
    print(f"핀 이미지 {n}개 확인")

In [11]:
# 브라우저 JS: 핀 URL·좌표·srcset → originals 고해상도 추출
EXTRACT_PINS_JS = """
const pins = document.querySelectorAll('div[data-test-id="pin-with-alt-text"]');
return Array.from(pins).map(pin => {
  const img = pin.querySelector('img');
  if (!img) return null;
  const srcset = img.srcset || '';
  const parts = srcset.split(',')
    .map(s => s.trim().split(/\\s+/)[0])
    .filter(u => u.includes('pinimg.com'));
  let url = parts.length ? parts[parts.length - 1] : (img.src || '');
  if (!url.includes('pinimg.com')) return null;
  url = url.replace(/\\/\\d+x\\//, '/originals/');
  const r = pin.getBoundingClientRect();
  return { url, top: r.top, left: r.left, width: r.width };
}).filter(Boolean);
"""


def get_pin_items(driver) -> list[dict]:
    """DOM에서 핀 목록(URL + 화면 좌표) 추출."""
    return driver.execute_script(EXTRACT_PINS_JS)


def _unique_by_url(items: list[dict]) -> list[dict]:
    """URL 중복 제거, 위→아래·왼→오른 순 정렬."""
    seen, out = set(), []
    for item in sorted(items, key=lambda x: (x["top"], x["left"])):
        if item["url"] not in seen:
            seen.add(item["url"])
            out.append(item)
    return out


def detect_column_count(items: list[dict]) -> int:
    """Pinterest masonry — 열(column) 개수 = 한 줄 핀 수."""
    if not items:
        return 1
    columns: list[float] = []
    for item in sorted(items, key=lambda x: x["left"]):
        if not columns or abs(item["left"] - columns[-1]) > COLUMN_X_TOLERANCE:
            columns.append(item["left"])
    return max(1, len(columns))


def detect_row_size(driver) -> int:
    """현재 화면의 열(한 줄) 개수."""
    return detect_column_count(_unique_by_url(get_pin_items(driver)))


def assign_column(item: dict, column_lefts: list[float]) -> int:
    """핀의 X좌표 → 열 인덱스."""
    return min(range(len(column_lefts)), key=lambda i: abs(item["left"] - column_lefts[i]))


def get_column_lefts(items: list[dict]) -> list[float]:
    """각 열의 X좌표 목록."""
    columns: list[float] = []
    for item in sorted(items, key=lambda x: x["left"]):
        if not columns or abs(item["left"] - columns[-1]) > COLUMN_X_TOLERANCE:
            columns.append(item["left"])
    return columns


def get_next_row_batch(driver, downloaded: set) -> list[str]:
    """맨 위 줄: 각 열에서 가장 위에 있는 미다운로드 핀."""
    items = _unique_by_url(get_pin_items(driver))
    pending = [x for x in items if x["url"] not in downloaded]
    if not pending:
        return []

    col_lefts = get_column_lefts(items)
    if not col_lefts:
        return [pending[0]["url"]]

    # 각 열별 최상단( top 최소 ) pending 핀
    by_col: dict[int, dict] = {}
    for item in pending:
        col = assign_column(item, col_lefts)
        if col not in by_col or item["top"] < by_col[col]["top"]:
            by_col[col] = item

    if not by_col:
        return []

    row_top = min(x["top"] for x in by_col.values())
    batch = [
        by_col[c]["url"]
        for c in sorted(by_col)
        if by_col[c]["top"] <= row_top + ROW_Y_TOLERANCE  # 같은 줄(Y) 허용 범위
    ]
    return batch


def get_all_pin_urls(driver) -> list[str]:
    """화면에 보이는 모든 핀 URL."""
    return [x["url"] for x in _unique_by_url(get_pin_items(driver))]


def scroll_once(driver, pause=SCROLL_PAUSE) -> bool:
    """한 칸 스크롤, 새 핀 로드 여부 반환."""
    before = len(get_all_pin_urls(driver))
    driver.execute_script("window.scrollBy(0, window.innerHeight);")
    time.sleep(pause)
    after = len(get_all_pin_urls(driver))
    return after > before


def download_one(session, url, save_dir=SAVE_DIR):
    """URL에서 이미지 1장 다운로드 (MD5 해시 파일명)."""
    r = session.get(url, timeout=20)
    r.raise_for_status()
    ext = Path(urlparse(url).path).suffix or ".jpg"
    name = hashlib.md5(url.encode()).hexdigest()[:12] + ext
    path = save_dir / name
    if path.exists():
        return name, False
    path.write_bytes(r.content)
    return name, True


def crawl_board(driver, save_dir=SAVE_DIR, max_count=MAX_IMAGES):
    """Pinterest masonry — 위쪽 한 줄씩 다운로드, 없으면 스크롤."""
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://kr.pinterest.com/",
    })

    row_size = detect_row_size(driver)  # 창 너비에 따라 열 수 자동 감지
    print(f"감지된 열(한 줄) 수: {row_size}개 — 창 너비에 따라 자동 변경")

    downloaded: set[str] = set()
    saved = skipped = row_num = no_new_scrolls = 0

    while saved < max_count:
        batch = get_next_row_batch(driver, downloaded)

        if not batch:
            print("다운로드할 이미지 없음 → 스크롤")
            if not scroll_once(driver):
                no_new_scrolls += 1
                if no_new_scrolls >= 3:
                    print("더 이상 새 핀이 없습니다.")
                    break
            else:
                no_new_scrolls = 0
                print(f"  스크롤 후 열 수: {detect_row_size(driver)}개")
            continue

        no_new_scrolls = 0
        row_num += 1
        print(f"\n[{row_num}줄] {len(batch)}장 처리 (열 {row_size}개 기준)")

        for url in batch:
            if saved >= max_count:
                break
            try:
                name, is_new = download_one(session, url, save_dir)
                downloaded.add(url)
                if is_new:
                    saved += 1
                    print(f"  저장: {name} ({saved}/{max_count})")
                else:
                    skipped += 1
                    print(f"  건너뜀(이미 있음): {name}")
            except Exception as e:
                downloaded.add(url)
                print(f"  실패: {url[:70]}... ({e})")

    print(f"\n신규 {saved}장 저장, {skipped}장 건너뜀 → {save_dir.resolve()}")

In [12]:
# === 실행 ===
driver = create_driver()

try:
    driver.get(BOARD_URL)  # Pinterest 보드 열기
    wait_for_pins(driver)  # 핀 로딩 대기
    print(f"보드 로드: {driver.title}")
    crawl_board(driver)  # masonry 그리드 한 줄씩 다운로드
finally:
    driver.quit()  # 브라우저 종료

보드 로딩 대기... (최대 30초)
핀 이미지 14개 확인
보드 로드: 490 원룸 인테리어 및 작은 방 아이디어 찾기 | 작은 방 디자인, 침실 꾸미기, 침실 디자인 등
감지된 열(한 줄) 수: 7개 — 창 너비에 따라 자동 변경

[1줄] 7장 처리 (열 7개 기준)
  저장: 2dcaff7b335a.jpg (1/300)
  저장: 6a9a167c50b8.jpg (2/300)
  저장: 59a13b47c446.jpg (3/300)
  저장: 53cc10273ae8.jpg (4/300)
  저장: d6794a6ca1ab.jpg (5/300)
  저장: 4fcdc799ed55.jpg (6/300)
  저장: 5b396f1331ac.jpg (7/300)

[2줄] 1장 처리 (열 7개 기준)
  저장: a48333b83d00.jpg (8/300)

[3줄] 2장 처리 (열 7개 기준)
  저장: 41965946f2e9.jpg (9/300)
  저장: b2308ef96596.png (10/300)

[4줄] 1장 처리 (열 7개 기준)
  저장: 4e58d9ab382a.jpg (11/300)

[5줄] 1장 처리 (열 7개 기준)
  저장: dca0f36204ac.jpg (12/300)

[6줄] 3장 처리 (열 7개 기준)
  저장: ec5a39f17b5d.jpg (13/300)
  저장: 47a009def130.jpg (14/300)
  저장: 53df45ad6d44.jpg (15/300)

[7줄] 2장 처리 (열 7개 기준)
  저장: 2762eb422fa2.jpg (16/300)
  저장: e9a249542285.jpg (17/300)

[8줄] 2장 처리 (열 7개 기준)
  저장: 9361fd94bdd3.jpg (18/300)
  저장: b21d42c9ceb4.jpg (19/300)

[9줄] 2장 처리 (열 7개 기준)
  저장: 413b26f113b4.jpg (20/300)
  저장: 9d8248c2dc24.jpg (21/300)

[10줄] 2